# Workshop 5 — Generative Adversarial Networks (GANs)

**Equipo 7 — Universidad Distrital Francisco José de Caldas**

Este cuaderno orquesta todo el Workshop 5 sobre el dataset MovieLens 32M:

1. GAN vanilla (BCE + label smoothing)
2. WGAN-GP (crítico 5:1 + gradient penalty)
3. cGAN condicional en `like` (opcional)
4. Evaluación de calidad (métricas de dominio) y modos de fallo
5. Muestras reales vs generadas
6. Experimento de data augmentation (4 condiciones) + tabla acumulativa W1–W5

Trabajamos sobre los **vectores de interacción 64-D** reutilizados de Workshop 4 (mismo split cronológico, `SEED=42`, sin fuga de datos).

---
# GUÍA KAGGLE PASO A PASO (para quien nunca ha usado Kaggle)

El entrenamiento de 3 GANs por 100+ épocas es lento en CPU. En una GPU de Kaggle tarda pocos minutos y es **gratis**. Sigue esto con calma, no te saltes pasos.

## Paso 0 — Crear una cuenta
1. Entra a **https://www.kaggle.com** y pulsa **Register** (puedes usar tu cuenta de Google).
2. La primera vez que quieras usar GPU, Kaggle te pedirá **verificar tu número de teléfono** (Settings → Phone verification). Es obligatorio para GPU. Hazlo una vez.

## Paso 1 — Subir el dataset de MovieLens y tu código
Kaggle no tiene acceso a tu disco; hay que subir los datos. Tienes dos opciones:

**Opción A (recomendada): subir el repo como *Dataset* de Kaggle.**
1. Comprime en tu PC la carpeta del repo `Machine_Learning` en un `.zip` **pero sin** la carpeta `.venv` ni `.git` (son pesadas e innecesarias). Asegúrate de incluir `src/`, `checkpoints/model_best.pt`, y la carpeta `dataSet/` con `ratings.csv` y `movies.csv`.
   - Si el `.zip` con `ratings.csv` (≈877 MB) es muy grande, puedes subir en su lugar el **MovieLens 32M oficial** desde Kaggle (ver Opción B) y subir el código aparte.
2. En Kaggle, menú izquierdo → **Datasets** → botón **+ New Dataset**.
3. Arrastra el `.zip`, ponle un título (p. ej. `ml-workshop`), y pulsa **Create**. Espera a que procese.

**Opción B: usar el MovieLens 32M ya publicado en Kaggle + subir solo el código.**
1. Sube solo tu código (carpeta `src/` + `checkpoints/model_best.pt`) como un Dataset pequeño llamado `ml-workshop-code`.
2. Más adelante añadirás además un dataset público de MovieLens 32M.

## Paso 2 — Crear el Notebook
1. Menú izquierdo → **Code** → **+ New Notebook**.
2. Arriba a la derecha pulsa **File → Import Notebook** y sube este archivo `w5_gan.ipynb`. (O copia las celdas a mano.)

## Paso 3 — Adjuntar tus datos al notebook
1. En el panel derecho del notebook, sección **Input**, pulsa **+ Add Input**.
2. Busca el dataset que subiste (`ml-workshop`) y pulsa **Add**. Aparecerá montado en `/kaggle/input/ml-workshop/`.

## Paso 4 — Activar la GPU
1. Panel derecho → sección **Session options** (o el menú **⋮ / Settings**).
2. En **Accelerator** elige **GPU T4 x2** (o la GPU que ofrezcan).
3. Confirma. Verás que la sesión se reinicia con GPU. Tienes ~30 h de GPU gratis por semana.

## Paso 5 — Ejecutar
1. Ejecuta las celdas de abajo **en orden** (Shift+Enter en cada una, o **Run All**).
2. La primera celda de setup localiza tu código y los datos; **ajusta la ruta** si tu dataset tiene otro nombre (te explico dónde).
3. Al terminar, los resultados quedan en `/kaggle/working/.../runs/workshop5/`. Para descargarlos: panel derecho → **Output** → botón de descarga, o `Save Version` para guardarlos de forma permanente.

> **Consejo:** si algo falla por memoria o tiempo, usa el modo rápido (`FAST=True` en la celda de configuración) para una prueba; luego ponlo en `False` para el run completo del informe.

---
## 1. Setup del entorno (Kaggle o local)

Esta celda detecta si estás en Kaggle o en local, localiza la carpeta del repo (donde está `src/`), y la pone en el `PYTHONPATH`. **Lee los comentarios y ajusta `KAGGLE_INPUT_NAME` si tu dataset se llama distinto.**

In [ ]:
import os, sys, shutil
from pathlib import Path

# ¿Estamos en Kaggle? (Kaggle define esta carpeta)
ON_KAGGLE = Path('/kaggle').exists()
print('Entorno:', 'KAGGLE' if ON_KAGGLE else 'LOCAL')

if ON_KAGGLE:
    # <-- AJUSTA ESTO: el nombre de la carpeta de tu dataset subido (mira en /kaggle/input/)
    KAGGLE_INPUT_NAME = 'ml-workshop'
    print('Contenido de /kaggle/input/:', os.listdir('/kaggle/input'))

    # buscamos la carpeta que contiene 'src' dentro del input (el repo puede estar anidado)
    input_root = Path('/kaggle/input') / KAGGLE_INPUT_NAME
    repo_src = None
    for p in input_root.rglob('src'):
        if (p / 'gan.py').exists() or (p / 'data_loader.py').exists():
            repo_src = p.parent
            break
    assert repo_src is not None, 'No encontré la carpeta src dentro del dataset. Revisa KAGGLE_INPUT_NAME.'

    # /kaggle/input es de SOLO LECTURA. Copiamos el repo a /kaggle/working para poder escribir
    # checkpoints, runs y la cache.
    REPO = Path('/kaggle/working/Machine_Learning')
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.copytree(repo_src, REPO)
    print('Repo copiado a:', REPO)
else:
    # local: este notebook vive en notebooks/, el repo es la carpeta de arriba
    REPO = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()

sys.path.insert(0, str(REPO))
os.chdir(REPO)
print('Carpeta de trabajo:', Path.cwd())
print('¿Existe src/gan.py?', (REPO / 'src' / 'gan.py').exists())

### 1b. Comprobar GPU y dependencias
En Kaggle, `torch`, `numpy`, `pandas`, `scikit-learn`, `scipy` y `matplotlib` ya vienen instalados. Si algo faltara, la celda lo instala.

In [ ]:
try:
    import torch, numpy, pandas, sklearn, scipy, matplotlib
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'numpy', 'pandas', 'scikit-learn', 'scipy', 'matplotlib', 'seaborn', 'tensorboard', 'tqdm'])
    import torch

print('torch', torch.__version__, '| GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('AVISO: sin GPU. Activa el Accelerator GPU (Paso 4) o usa FAST=True abajo.')

### 1c. Comprobar que el dataset y el checkpoint de W2 están presentes
El pipeline necesita `dataSet/ratings.csv`, `dataSet/movies.csv` y `checkpoints/model_best.pt` (el modelo entrenado de Workshop 2).

In [ ]:
need = ['dataSet/ratings.csv', 'dataSet/movies.csv', 'checkpoints/model_best.pt']
for f in need:
    p = REPO / f
    print(('OK  ' if p.exists() else 'FALTA'), f, f'({p.stat().st_size/1e6:.1f} MB)' if p.exists() else '')
print('\nSi falta ratings.csv, añade el MovieLens 32M como Input y copia los CSV a dataSet/.')

---
## 2. Configuración (modo rápido vs completo)

- `FAST=True`: pocas épocas y pocos datos, para una prueba de humo de 1–2 minutos.
- `FAST=False`: run completo (120 épocas) para los resultados del informe.

Esta celda sólo fija unas variables que usaremos al llamar a las funciones.

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FAST = False          # <-- ponlo en True para una prueba rápida
N_EPOCHS = 15 if FAST else 120
MAX_TRAIN = 30_000 if FAST else 200_000
print(f'device={device} | FAST={FAST} | epocas={N_EPOCHS} | max_train={MAX_TRAIN}')

---
## 3. Preparar los datos 64-D (reutiliza Workshop 4)

La primera vez construye la cache (puede tardar un par de minutos porque carga el modelo de W2 y arma los vectores). Luego lee de cache al instante.

In [ ]:
from src.data_w5 import load_w5_data
data = load_w5_data(use_cache=True, max_train_normal=MAX_TRAIN, device=device)
for k, v in data.items():
    print(f'{k:20s} {getattr(v, "shape", None)}')

---
## 4. Entrenar la GAN vanilla y la WGAN-GP

In [ ]:
from src.train_gan import train_vanilla_gan, train_wgan_gp
x_train = data['X_train_scaled']
_ = train_vanilla_gan(x_train, n_epochs=N_EPOCHS, device=device)
_ = train_wgan_gp(x_train, n_epochs=N_EPOCHS, device=device)

## 5. Entrenar la cGAN (opcional, condicionada en `like`)

In [ ]:
from src.cgan import train_cgan
_ = train_cgan(data['X_val_all_scaled'], data['y_val'], n_epochs=N_EPOCHS, device=device)

---
## 6. Evaluar calidad de la generación (métricas de dominio + modos de fallo)

In [ ]:
from src.evaluate_gan import main as eval_main
quality = eval_main(n_samples=3000 if FAST else 5000)

## 7. Generar muestras y figuras (real vs generado)

In [ ]:
from src.generate_samples import main as gen_main
gen_main(n_samples=3000)

# mostramos las figuras generadas dentro del notebook
from IPython.display import Image, display
runs = REPO / 'runs' / 'workshop5'
for img in ['grid_real_vs_vanilla.png', 'grid_real_vs_wgan_gp.png', 'pca_real_vs_generated.png']:
    p = runs / img
    if p.exists():
        print(img); display(Image(str(p)))

---
## 8. Experimento de data augmentation (4 condiciones)

(a) real · (b) real+vanilla · (c) real+WGAN-GP · (d) real+cGAN balanceado.

In [ ]:
from src.augment_and_retrain import main as augment_main
aug = augment_main(n_synth=3000)

## 9. Tabla acumulativa W1–W5

In [ ]:
from src.cumulative_table import main as table_main
df = table_main()
df

---
## 10. Descargar los resultados

Todo quedó en `runs/workshop5/` y `checkpoints/`. En Kaggle:
- **Save Version** (arriba a la derecha) guarda el notebook y su `Output` de forma permanente.
- O en el panel derecho → **Output** → descarga las carpetas.

Archivos clave para el informe PDF (`group7_workshop5.pdf`):
- `runs/workshop5/gan_quality_metrics.json` — métricas de calidad vanilla vs WGAN-GP
- `runs/workshop5/augmentation_results.json` — resultados del augmentation
- `runs/workshop5/cumulative_comparison.csv` — tabla W1–W5
- `runs/workshop5/grid_real_vs_*.png`, `feature_dist_*.png`, `pca_real_vs_generated.png` — figuras
- `checkpoints/gan_vanilla_best.pt`, `gan_wgan_best.pt`, `cgan_best.pt` — generadores entrenados